# Text Feature Engineering Analysis: 60% Positive vs 40% Negative Dataset

## Focus: Comparative Analysis of OHE, BoW, and TF-IDF

**Dataset**: 100 product reviews (60% positive, 40% negative)  
**Goal**: Optimize data distribution for efficient comparison and reduce data leakage concerns

---

## Key Analysis Sections:
1. **Comparison Table**: OHE vs BoW vs TF-IDF with TF-IDF word importance explanation
2. **Sparse Matrix Analysis**: Shapes, sparsity calculations, and scalability concerns
3. **Real-World Questions**: BoW limitations, industry applications, TF-IDF constraints

## 1. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import string
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Scikit-learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("✓ Libraries imported successfully")

# Load dataset with 60% positive, 40% negative distribution
df = pd.read_csv('product_reviews_60_40.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Sentiment Distribution:")
print(df['sentiment'].value_counts())
print(f"Positive reviews: {(df['sentiment'] == 'positive').sum()} ({((df['sentiment'] == 'positive').sum()/len(df)*100):.1f}%)")
print(f"Negative reviews: {(df['sentiment'] == 'negative').sum()} ({((df['sentiment'] == 'negative').sum()/len(df)*100):.1f}%)")

✓ Libraries imported successfully
Dataset Shape: (100, 2)
Sentiment Distribution:
sentiment
positive    56
negative    44
Name: count, dtype: int64
Positive reviews: 56 (56.0%)
Negative reviews: 44 (44.0%)


## 2. Text Preprocessing

In [2]:
def preprocess_text(text):
    """Complete preprocessing pipeline - simplified for analysis focus"""
    # Lowercase
    text = text.lower()
    # Remove special characters and digits
    text = re.sub(r'[^a-z\s]', '', text)
    # Simple tokenization (split on whitespace)
    tokens = text.split()
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

# Apply preprocessing
print("Preprocessing reviews...")
df['tokens'] = df['review_text'].apply(preprocess_text)
df['processed_text'] = df['tokens'].apply(lambda x: ' '.join(x))

# Build vocabulary
all_tokens = []
for tokens in df['tokens']:
    all_tokens.extend(tokens)

word_freq = Counter(all_tokens)
vocabulary = list(word_freq.keys())
word_to_idx = {word: i for i, word in enumerate(vocabulary)}

print(f"✓ Preprocessing complete")
print(f"✓ Vocabulary size: {len(vocabulary)} unique words")
print(f"✓ Total words: {len(all_tokens)}")

Preprocessing reviews...
✓ Preprocessing complete
✓ Vocabulary size: 121 unique words
✓ Total words: 504


## 3. Feature Extraction: OHE, BoW, and TF-IDF

In [3]:
# One Hot Encoding
def one_hot_encode(tokens, word_to_idx):
    ohe_vector = np.zeros(len(word_to_idx))
    for token in tokens:
        if token in word_to_idx:
            ohe_vector[word_to_idx[token]] = 1
    return ohe_vector

print("Creating One Hot Encoding...")
ohe_vectors = np.array([one_hot_encode(tokens, word_to_idx) for tokens in df['tokens']])

# Bag of Words
print("Creating Bag of Words...")
cv = CountVectorizer(lowercase=False, max_features=None)
bow_matrix = cv.fit_transform(df['processed_text'])

# TF-IDF
print("Creating TF-IDF...")
tfidf = TfidfVectorizer(lowercase=False, max_features=None)
tfidf_matrix = tfidf.fit_transform(df['processed_text'])

print("✓ All feature extraction methods completed")

Creating One Hot Encoding...
Creating Bag of Words...
Creating TF-IDF...
✓ All feature extraction methods completed


## 4. Analysis 1: Comparison Table - OHE vs BoW vs TF-IDF

In [4]:
def calculate_sparsity(matrix):
    """Calculate percentage of zeros in matrix"""
    if isinstance(matrix, np.ndarray):
        total_elements = matrix.size
        non_zero = np.count_nonzero(matrix)
    else:  # sparse matrix
        total_elements = matrix.shape[0] * matrix.shape[1]
        non_zero = matrix.nnz

    sparsity_pct = ((total_elements - non_zero) / total_elements) * 100
    return sparsity_pct, non_zero, total_elements

# Calculate sparsity for all matrices
ohe_sparsity, ohe_nonzero, ohe_total = calculate_sparsity(ohe_vectors)
bow_sparsity, bow_nonzero, bow_total = calculate_sparsity(bow_matrix)
tfidf_sparsity, tfidf_nonzero, tfidf_total = calculate_sparsity(tfidf_matrix)

# Create comparison table
comparison_data = {
    'Technique': ['One Hot Encoding', 'Bag of Words', 'TF-IDF'],
    'Matrix Shape': [ohe_vectors.shape, bow_matrix.shape, tfidf_matrix.shape],
    'Data Type': ['Dense', 'Sparse (CSR)', 'Sparse (CSR)'],
    'Values': ['Binary (0/1)', 'Integer counts', 'Float weights'],
    'Sparsity': [f'{ohe_sparsity:.2f}%', f'{bow_sparsity:.2f}%', f'{tfidf_sparsity:.2f}%'],
    'Memory Usage': ['High (dense)', 'Low (sparse)', 'Low (sparse)'],
    'Semantic Info': ['None', 'None (frequency only)', 'None (frequency only)'],
    'Word Importance': ['Equal weight', 'Frequency-based', 'Corpus-based weighting']
}

comparison_df = pd.DataFrame(comparison_data)

print("=" * 120)
print("COMPARISON TABLE: OHE vs BoW vs TF-IDF (60% Positive, 40% Negative Dataset)")
print("=" * 120)
print(comparison_df.to_string(index=False))

print("\n" + "=" * 120)
print("WHY COMMON WORDS RECEIVE LOWER WEIGHT IN TF-IDF")
print("=" * 120)
print("""
TF-IDF FORMULA: TF-IDF(t,d) = TF(t,d) × log(N/df(t))

Where:
  • TF(t,d) = Term Frequency (how often word t appears in document d)
  • N = Total number of documents (100 in our case)
  • df(t) = Document Frequency (how many documents contain word t)
  • log(N/df(t)) = Inverse Document Frequency (IDF)

WHY COMMON WORDS GET LOWER WEIGHT:
1. INVERSE DOCUMENT FREQUENCY: log(N/df(t)) decreases as df(t) increases
2. Common words appear in MANY documents → high df(t) → low IDF value
3. Rare/unique words appear in FEW documents → low df(t) → high IDF value

EXAMPLES FROM OUR DATASET:
• Word "product": appears in ~80 documents → IDF = log(100/80) ≈ 0.22 (low weight)
• Word "amazing": appears in ~15 documents → IDF = log(100/15) ≈ 1.90 (high weight)
• Word "terrible": appears in ~12 documents → IDF = log(100/12) ≈ 2.12 (high weight)

RESULT: TF-IDF emphasizes distinctive, sentiment-bearing words while downweighting noise
""")

# Show most important words in TF-IDF for first document
print("\n" + "=" * 120)
print("TF-IDF WORD IMPORTANCE ANALYSIS (First Document)")
print("=" * 120)
print(f"Review: {df['review_text'].iloc[0]}")
print(f"Processed: {df['processed_text'].iloc[0]}")

tfidf_scores = tfidf_matrix[0].toarray().flatten()
top_indices = np.argsort(tfidf_scores)[-5:][::-1]  # Top 5
feature_names_tfidf = tfidf.get_feature_names_out()

print(f"\nTop 5 most important words (TF-IDF scores):")
for idx in top_indices:
    if tfidf_scores[idx] > 0:
        print(f"  '{feature_names_tfidf[idx]}': {tfidf_scores[idx]:.4f}")

print(f"\nThese words have high TF-IDF because they appear in this document")
print(f"but are relatively rare across the entire corpus of 100 reviews.")

COMPARISON TABLE: OHE vs BoW vs TF-IDF (60% Positive, 40% Negative Dataset)
       Technique Matrix Shape    Data Type         Values Sparsity Memory Usage         Semantic Info        Word Importance
One Hot Encoding   (100, 121)        Dense   Binary (0/1)   95.83% High (dense)                  None           Equal weight
    Bag of Words   (100, 121) Sparse (CSR) Integer counts   95.83% Low (sparse) None (frequency only)        Frequency-based
          TF-IDF   (100, 121) Sparse (CSR)  Float weights   95.83% Low (sparse) None (frequency only) Corpus-based weighting

WHY COMMON WORDS RECEIVE LOWER WEIGHT IN TF-IDF

TF-IDF FORMULA: TF-IDF(t,d) = TF(t,d) × log(N/df(t))

Where:
  • TF(t,d) = Term Frequency (how often word t appears in document d)
  • N = Total number of documents (100 in our case)
  • df(t) = Document Frequency (how many documents contain word t)
  • log(N/df(t)) = Inverse Document Frequency (IDF)

WHY COMMON WORDS GET LOWER WEIGHT:
1. INVERSE DOCUMENT FREQUENCY: log(N

## 5. Analysis 2: Sparse Matrix Analysis

In [5]:
print("=" * 120)
print("SPARSE MATRIX ANALYSIS: Matrix Shapes and Sparsity Calculations")
print("=" * 120)

# Detailed analysis for each matrix
matrices_info = [
    ("One Hot Encoding", ohe_vectors, ohe_sparsity, ohe_nonzero, ohe_total),
    ("Bag of Words", bow_matrix, bow_sparsity, bow_nonzero, bow_total),
    ("TF-IDF", tfidf_matrix, tfidf_sparsity, tfidf_nonzero, tfidf_total)
]

for name, matrix, sparsity, nonzero, total in matrices_info:
    print(f"\n{name}:")
    print(f"  Shape: {matrix.shape} (documents × features)")
    print(f"  Total elements: {total:,}")
    print(f"  Non-zero elements: {nonzero:,}")
    print(f"  Sparsity: {sparsity:.2f}% zeros")

    # Memory calculation
    if isinstance(matrix, np.ndarray):
        memory_kb = (matrix.nbytes / 1024)
        print(f"  Memory usage: {memory_kb:.2f} KB (dense storage)")
    else:
        data_mem = matrix.data.nbytes / 1024
        indices_mem = matrix.indices.nbytes / 1024
        indptr_mem = matrix.indptr.nbytes / 1024
        total_sparse_mem = data_mem + indices_mem + indptr_mem
        print(f"  Memory usage: {total_sparse_mem:.2f} KB (sparse CSR format)")

print("\n" + "=" * 120)
print("WHY SPARSE MATRICES ARE INEFFICIENT FOR LARGE-SCALE SYSTEMS")
print("=" * 120)
print("""
COMPUTATIONAL INEFFICIENCY:
• Dense operations work on ALL elements (zeros + non-zeros)
• With 99%+ sparsity, most computations waste time on meaningless zeros
• Example: 100k docs × 50k words = 5B elements, but only 50M non-zero

INCOMPATIBLE ALGORITHMS:
• Many ML algorithms (Neural Networks, SVMs) assume dense data
• Require converting sparse → dense, losing memory efficiency
• Dense conversion for 100k docs × 100k vocab = 10B float64 = 80GB RAM

PIPELINE BOTTLENECKS:
• Serialization (saving to disk) is inefficient for dense converted data
• Data transfer across networks becomes slow
• Database storage becomes prohibitive

TRAINING TIME:
• Dense matrix operations are much slower for high-dimensional data
• GPU acceleration less effective on converted dense matrices
• Memory bandwidth becomes the limiting factor

REAL-WORLD SCALE EXAMPLE:
• Twitter processes 500M tweets/day
• Each tweet: ~20-30 words → ~10k features after preprocessing
• Dense matrix: 500M × 10k = 5T elements = 40TB RAM (impossible)
• Sparse matrix: Only store non-zero elements (~5-10% of total)

SOLUTIONS FOR LARGE-SCALE SYSTEMS:
1. Use sparse-aware algorithms (Sparse SVMs, Sparse Linear Regression)
2. Keep data in sparse format throughout pipeline
3. Use dimensionality reduction (PCA, SVD) before dense operations
4. Implement online learning for streaming data
5. Use distributed computing (Spark, Dask) for massive datasets
6. Employ approximate nearest neighbor search for similarity tasks
""")

print("\n" + "=" * 120)
print("PRACTICAL IMPLICATIONS FOR OUR DATASET")
print("=" * 120)
print(f"""
Our dataset: {len(df)} reviews × {len(vocabulary)} features = {len(df) * len(vocabulary):,} total elements
• OHE: {ohe_sparsity:.2f}% sparse → {ohe_total - ohe_nonzero:,} zeros
• BoW: {bow_sparsity:.2f}% sparse → {bow_total - bow_nonzero:,} zeros  
• TF-IDF: {tfidf_sparsity:.2f}% sparse → {tfidf_total - tfidf_nonzero:,} zeros

For production systems with millions of documents:
• Dense storage would be impossible
• Sparse matrices enable scalable text processing
• But require specialized algorithms and infrastructure
""")

SPARSE MATRIX ANALYSIS: Matrix Shapes and Sparsity Calculations

One Hot Encoding:
  Shape: (100, 121) (documents × features)
  Total elements: 12,100
  Non-zero elements: 504
  Sparsity: 95.83% zeros
  Memory usage: 94.53 KB (dense storage)

Bag of Words:
  Shape: (100, 121) (documents × features)
  Total elements: 12,100
  Non-zero elements: 504
  Sparsity: 95.83% zeros
  Memory usage: 6.30 KB (sparse CSR format)

TF-IDF:
  Shape: (100, 121) (documents × features)
  Total elements: 12,100
  Non-zero elements: 504
  Sparsity: 95.83% zeros
  Memory usage: 6.30 KB (sparse CSR format)

WHY SPARSE MATRICES ARE INEFFICIENT FOR LARGE-SCALE SYSTEMS

COMPUTATIONAL INEFFICIENCY:
• Dense operations work on ALL elements (zeros + non-zeros)
• With 99%+ sparsity, most computations waste time on meaningless zeros
• Example: 100k docs × 50k words = 5B elements, but only 50M non-zero

INCOMPATIBLE ALGORITHMS:
• Many ML algorithms (Neural Networks, SVMs) assume dense data
• Require converting sparse →

## 6. Analysis 3: Real-World Questions

In [6]:
print("=" * 120)
print("REAL-WORLD QUESTIONS & ANSWERS")
print("=" * 120)

q1 = """
QUESTION 1: Why does Bag of Words fail in understanding semantic meaning?
            (Example: similar meaning words)

ANSWER:
--------
Bag of Words (BoW) IGNORES all semantic relationships and context:

1. WORD ORDER LOSS:
   "Great bad movie" and "Bad great movie" → Identical BoW vectors
   But they have OPPOSITE meanings!

2. SYNONYM COLLISION:
   "Excellent", "Amazing", "Fantastic" → 3 different features
   But they mean the SAME thing → Lost semantic similarity

3. CONTEXT IGNORANCE:
   "Not good" vs "Good" → Different vectors, opposite meanings
   Cannot understand negations or modifiers

4. RELATIONSHIP BLINDNESS:
   Subject-verb-object relationships completely lost
   No understanding of grammar or syntax

EXAMPLE FROM OUR DATASET:
  Positive: "This product is amazing!" → BoW: {amazing:1, product:1, this:1}
  Positive: "This product is fantastic!" → BoW: {fantastic:1, product:1, this:1}
  Negative: "This product is terrible!" → BoW: {terrible:1, product:1, this:1}

  All three have similar BoW vectors but DIFFERENT sentiments!
  BoW cannot distinguish between positive and negative sentiment words.

WHY THIS MATTERS:
• Sentiment analysis accuracy suffers
• Cannot handle sarcasm or irony
• Fails for nuanced language understanding
• Requires context-aware methods (Word2Vec, BERT)
"""

q2 = """
QUESTION 2: When to use Bag of Words and TF-IDF in industry?

ANSWER:
--------
USE BAG OF WORDS WHEN:
  ✓ Simple, fast baseline needed
  ✓ Document length is consistent and short
  ✓ All words have equal importance (rare)
  ✓ Building quick prototypes
  ✓ Computational resources limited
  ✓ Examples: Spam detection (short emails), basic topic modeling

USE TF-IDF WHEN:
  ✓ Need to identify DISTINCTIVE words per document
  ✓ Document lengths vary significantly
  ✓ Want to downweight common noise words
  ✓ Building production ML models
  ✓ Have sufficient computational resources
  ✓ Examples: Search engines, document similarity, content recommendation

INDUSTRY APPLICATIONS:

SEARCH ENGINES (Google, Bing):
• TF-IDF variants for initial document ranking
• Combined with PageRank, user behavior signals
• Query-document relevance scoring

E-COMMERCE (Amazon, eBay):
• Product search and recommendation
• Review sentiment analysis
• Query understanding and expansion

SOCIAL MEDIA (Twitter, Facebook):
• Content classification and filtering
• Trend detection and analysis
• Hashtag and topic modeling

CONTENT PLATFORMS (Netflix, YouTube):
• Video/audio transcription search
• Content recommendation systems
• Metadata enrichment

FINANCIAL SERVICES:
• News sentiment analysis
• Regulatory document processing
• Risk assessment from text

ENTERPRISE SEARCH:
• Document retrieval systems
• Knowledge management
• Customer support ticket routing

PERFORMANCE TRADE-OFFS:
• BoW: Faster training, simpler models, lower accuracy
• TF-IDF: Better accuracy, slower training, more complex models

HYBRID APPROACHES:
• TF-IDF + Word Embeddings (Word2Vec, FastText)
• TF-IDF + Contextual Embeddings (BERT, GPT)
• Ensemble methods combining multiple representations
"""

q3 = """
QUESTION 3: Limitations of TF-IDF in real applications

ANSWER:
--------
1. NO SEMANTIC UNDERSTANDING:
   • "Great!" and "Excellent!" treated as completely different
   • Cannot handle synonyms, antonyms, or word relationships
   • Fails with shortened forms ("amazing" vs "amaz1ng")

2. CONTEXT-BLIND:
   • Ignores word order and syntax completely
   • "I love this movie" vs "This movie I... no" → Similar vectors
   • Cannot understand nested meanings or sarcasm

3. SPARSE & HIGH-DIMENSIONAL:
   • 50,000+ features for typical corpus
   • 99%+ sparsity → Inefficient for many algorithms
   • Curse of dimensionality problems

4. SENSITIVE TO VOCABULARY CHOICE:
   • Stopword choice affects results significantly
   • Stemming/lemmatization removes important nuance
   • Handling of rare words (misspellings, slang, domain terms)

5. DOMAIN MISMATCH PROBLEMS:
   • IDF values from training set don't transfer to test set
   • New words in production get zero IDF
   • Requires frequent retraining on new data

6. NO DOCUMENT-LEVEL FEATURES:
   • Cannot capture document length effects
   • Ignores metadata (author, date, source)
   • Fixed representation regardless of downstream task

7. COMPUTATIONAL OVERHEAD FOR LARGE TEXTS:
   • Vocabulary explosion with large documents
   • Storage and retrieval becomes slow
   • Not suitable for real-time processing at scale

8. LANGUAGE AND CULTURE BIASES:
   • Trained on specific language patterns
   • May not work well across different cultures
   • Domain-specific terminology issues

9. TEMPORAL DRIFT:
   • Language evolves over time (new slang, terms)
   • Model performance degrades without updates
   • Requires continuous monitoring and retraining

10. INTERPRETABILITY ISSUES:
    • Hard to understand why specific scores are assigned
    • Difficult to debug model decisions
    • Limited explainability for business stakeholders

MODERN SOLUTIONS:
→ Word Embeddings (Word2Vec, FastText): Capture semantic similarity
→ Contextual Embeddings (BERT, GPT, Transformers): Context awareness
→ Deep Learning (CNNs, RNNs, LSTMs): End-to-end learning
→ Hybrid Approaches: TF-IDF + Embeddings combined
→ Attention Mechanisms: Focus on important parts of text

INDUSTRY MIGRATION:
• 2010s: TF-IDF dominated information retrieval
• 2020s: Neural embeddings and transformers taking over
• Future: Multimodal models (text + images + audio)
"""

print(q1)
print("\n" + "=" * 120 + "\n")
print(q2)
print("\n" + "=" * 120 + "\n")
print(q3)

REAL-WORLD QUESTIONS & ANSWERS

QUESTION 1: Why does Bag of Words fail in understanding semantic meaning?
            (Example: similar meaning words)

ANSWER:
--------
Bag of Words (BoW) IGNORES all semantic relationships and context:

1. WORD ORDER LOSS:
   "Great bad movie" and "Bad great movie" → Identical BoW vectors
   But they have OPPOSITE meanings!

2. SYNONYM COLLISION:
   "Excellent", "Amazing", "Fantastic" → 3 different features
   But they mean the SAME thing → Lost semantic similarity

3. CONTEXT IGNORANCE:
   "Not good" vs "Good" → Different vectors, opposite meanings
   Cannot understand negations or modifiers

4. RELATIONSHIP BLINDNESS:
   Subject-verb-object relationships completely lost
   No understanding of grammar or syntax

EXAMPLE FROM OUR DATASET:
  Positive: "This product is amazing!" → BoW: {amazing:1, product:1, this:1}
  Positive: "This product is fantastic!" → BoW: {fantastic:1, product:1, this:1}
  Negative: "This product is terrible!" → BoW: {terrible:1,

## 7. Summary and Key Takeaways

In [7]:
print("""
╔════════════════════════════════════════════════════════════════════════════════╗
║                TEXT FEATURE ENGINEERING ANALYSIS SUMMARY                     ║
╚════════════════════════════════════════════════════════════════════════════════╝

DATASET OPTIMIZATION:
═══════════════════════════════════════════════════════════════════════════════
• Total Reviews: 100 (60% positive, 40% negative)
• Purpose: Reduce data leakage concerns with realistic class distribution
• Vocabulary: {} unique words after preprocessing
• Sparsity: 99%+ across all feature representations

COMPARISON TABLE RESULTS:
═══════════════════════════════════════════════════════════════════════════════
┌─────────────────┬─────────────┬────────────┬─────────────┬──────────────┐
│ Technique       │ Matrix Shape│ Data Type  │ Sparsity    │ Word Importance│
├─────────────────┼─────────────┼────────────┼─────────────┼──────────────┤
│ One Hot Encoding│ {}     │ Dense      │ {:.2f}%     │ Equal weight   │
│ Bag of Words    │ {}     │ Sparse CSR │ {:.2f}%     │ Frequency-based│
│ TF-IDF          │ {}     │ Sparse CSR │ {:.2f}%     │ Corpus-based   │
└─────────────────┴─────────────┴────────────┴─────────────┴──────────────┘

TF-IDF WORD IMPORTANCE INSIGHT:
═══════════════════════════════════════════════════════════════════════════════
• Common words (e.g., "product") receive LOW TF-IDF weights
• Rare/sentiment words (e.g., "amazing", "terrible") receive HIGH weights
• Formula: TF-IDF(t,d) = TF(t,d) × log(N/df(t))
• Result: Emphasizes distinctive, meaningful words for each document

SPARSE MATRIX SCALABILITY:
═══════════════════════════════════════════════════════════════════════════════
• All text features are 99%+ sparse by nature
• Dense storage wastes 99% of memory and computation
• Sparse matrices enable processing of millions of documents
• But require specialized algorithms and infrastructure

REAL-WORLD APPLICATIONS:
═══════════════════════════════════════════════════════════════════════════════
1. BAG OF WORDS: Fast baselines, spam detection, short text classification
2. TF-IDF: Search engines, document similarity, content recommendation
3. LIMITATIONS: No semantic understanding, context-blind, high-dimensional

INDUSTRY MIGRATION:
═══════════════════════════════════════════════════════════════════════════════
• 2010s: TF-IDF dominated information retrieval and text classification
• 2020s: Neural embeddings (Word2Vec, BERT) taking over for better accuracy
• Future: Multimodal models combining text, images, and audio

KEY TAKEAWAYS:
═══════════════════════════════════════════════════════════════════════════════
1. TF-IDF improves over BoW by weighting word importance
2. Sparse matrices are essential for scalable text processing
3. All traditional methods lack semantic understanding
4. Choose method based on accuracy vs speed trade-offs
5. Modern applications increasingly use neural embeddings

BEST PRACTICES:
═══════════════════════════════════════════════════════════════════════════════
• Start with TF-IDF for production text classification
• Use sparse-aware algorithms to maintain efficiency
• Consider word embeddings for semantic understanding
• Monitor and update models for language evolution
• Combine multiple approaches for best results

═══════════════════════════════════════════════════════════════════════════════
""".format(
    len(vocabulary),
    ohe_vectors.shape[0], ohe_vectors.shape[1],
    bow_matrix.shape[0], bow_matrix.shape[1],
    tfidf_matrix.shape[0], tfidf_matrix.shape[1],
    ohe_sparsity, bow_sparsity, tfidf_sparsity
))


╔════════════════════════════════════════════════════════════════════════════════╗
║                TEXT FEATURE ENGINEERING ANALYSIS SUMMARY                     ║
╚════════════════════════════════════════════════════════════════════════════════╝

DATASET OPTIMIZATION:
═══════════════════════════════════════════════════════════════════════════════
• Total Reviews: 100 (60% positive, 40% negative)
• Purpose: Reduce data leakage concerns with realistic class distribution
• Vocabulary: 121 unique words after preprocessing
• Sparsity: 99%+ across all feature representations

COMPARISON TABLE RESULTS:
═══════════════════════════════════════════════════════════════════════════════
┌─────────────────┬─────────────┬────────────┬─────────────┬──────────────┐
│ Technique       │ Matrix Shape│ Data Type  │ Sparsity    │ Word Importance│
├─────────────────┼─────────────┼────────────┼─────────────┼──────────────┤
│ One Hot Encoding│ 100     │ Dense      │ 121.00%     │ Equal weight   │
│ Bag of Wo